## Session **Spark**

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ETL_Melt_Pizza_Talca").getOrCreate()

**## Extraccion CSV & `Lectura`******

In [0]:
ruta = "/Volumes/melt_pizza/mel_pizza_raw/volumen_melt/Pedidos_Melt_Pizza_Talca.csv"
df_raw = spark.read.csv(ruta, header=True, inferSchema=True)
display(df_raw)

ID_Pedido,Nombre,Apellido,Telefono,Canal,Pizza,Ingredientes,Cantidad,Precio_Unitario,Fecha_Pedido,Hora_Pedido,Hora_Listo,Hora_Retiro,Minutos_Espera_Retiro,Estado_Retiro,Local
MP-TAL-1001,Natalia,Silva,+56 9 6102 5506,PedidosYa,Cuatro Quesos,"Mozzarella, parmesano, gorgonzola, queso crema",1,10990,2026-08-28,13:29:00,13:50:00,14:14,24,Retirado tarde,Talca
MP-TAL-1002,Valentina,Muñoz,+56 9 6383 4582,Presencial,Cuatro Quesos,"Mozzarella, parmesano, gorgonzola, queso crema",2,14990,2026-08-29,21:18:00,21:36:00,21:40,4,Retirado a tiempo,Talca
MP-TAL-1003,Amanda,Vera,+56 9 7139 1106,Presencial,Pepperoni Clásica,"Salsa de tomate, mozzarella, pepperoni",3,12990,2026-08-30,17:44:00,17:57:00,17:59,2,Retirado a tiempo,Talca
MP-TAL-1004,Florencia,Contreras,+56 9 7470 6635,Uber Eats,Vegetariana,"Salsa de tomate, mozzarella, champiñones, pimentón, cebolla morada, aceitunas",1,12990,2026-08-27,13:20:00,13:40:00,N/A,null,No retirado (perdido),Talca
MP-TAL-1005,Trinidad,Ortiz,+56 9 9626 6925,Presencial,Cuatro Quesos,"Mozzarella, parmesano, gorgonzola, queso crema",3,9990,2026-08-26,13:21:00,13:46:00,13:50,4,Retirado a tiempo,Talca
MP-TAL-1006,Javiera,Tapia,+56 9 7138 8428,Uber Eats,Hawaiana,"Salsa de tomate, mozzarella, jamón, piña",1,11990,2026-08-29,13:13:00,13:34:00,14:05,31,Retirado tarde,Talca
MP-TAL-1007,Constanza,Riquelme,+56 9 8986 5010,Presencial,Pepperoni Clásica,"Salsa de tomate, mozzarella, pepperoni",2,12990,2026-08-29,21:30:00,21:45:00,N/A,null,No retirado (perdido),Talca
MP-TAL-1008,Sergio,Castillo,+56 9 9452 1916,Uber Eats,Cuatro Quesos,"Mozzarella, parmesano, gorgonzola, queso crema",1,11990,2026-08-28,17:22:00,17:37:00,17:45,8,Retirado a tiempo,Talca
MP-TAL-1009,Agustín,Bravo,+56 9 6585 5339,Uber Eats,Pepperoni Clásica,"Salsa de tomate, mozzarella, pepperoni",1,14990,2026-08-27,21:57:00,22:15:00,22:37,22,Retirado tarde,Talca
MP-TAL-1010,Catalina,Martínez,+56 9 8087 9085,Presencial,Chilenaza,"Salsa de tomate, mozzarella, choricillo, cebolla, tomate, orégano",1,9990,2026-08-29,19:12:00,19:33:00,19:35,2,Retirado a tiempo,Talca


## Orden  5 primeros `clientes`

In [0]:
display(df_raw.orderBy(df_raw.ID_Pedido.desc()).limit(5))

ID_Pedido,Nombre,Apellido,Telefono,Canal,Pizza,Ingredientes,Cantidad,Precio_Unitario,Fecha_Pedido,Hora_Pedido,Hora_Listo,Hora_Retiro,Minutos_Espera_Retiro,Estado_Retiro,Local
Nota: Datos ficticios generados para el ejercicio de ETL (Databricks/Apache Spark). Representan el archivo Excel base que hoy reporta el local de Talca a la central de Santiago.,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
MP-TAL-1060,Carolina,Sánchez,+56 9 7904 8136,PedidosYa,Vegetariana,"Salsa de tomate, mozzarella, champiñones, pimentón, cebolla morada, aceitunas",1,10990,2026-08-27,16:09:00,16:33:00,16:41,8,Retirado a tiempo,Talca
MP-TAL-1059,Álvaro,Contreras,+56 9 7775 6934,Uber Eats,Prosciutto e Funghi,"Salsa de tomate, mozzarella, jamón serrano, champiñones, parmesano",3,10990,2026-08-29,16:36:00,16:57:00,17:33,36,Retirado tarde,Talca
MP-TAL-1058,Hernán,Valenzuela,+56 9 9482 2983,Uber Eats,Napoletana,"Salsa de tomate, mozzarella, tomate fresco, albahaca, aceite de oliva",1,14990,2026-08-28,17:49:00,18:01:00,18:08,7,Retirado a tiempo,Talca
MP-TAL-1057,Francisco,Muñoz,+56 9 9804 6111,Presencial,BBQ Pollo,"Salsa BBQ, mozzarella, pollo, cebolla morada, cilantro",2,10990,2026-08-29,16:08:00,16:21:00,16:54,33,Retirado tarde,Talca


## # Transformación: pedidos **perdidos**

In [0]:
from pyspark.sql.functions import col

df_perdidos = (
    df_raw
    .filter(col("Estado_Retiro") == "No retirado (perdido)")
    .select(
        "ID_Pedido", "Nombre", "Apellido", "Telefono",
        "Canal", "Pizza", "Fecha_Pedido", "Hora_Pedido",
        "Hora_Listo", "Estado_Retiro", "Local"
    )
    .orderBy(col("Fecha_Pedido").desc())
)

display(df_perdidos)
print("Total pedidos perdidos:", df_perdidos.count())

ID_Pedido,Nombre,Apellido,Telefono,Canal,Pizza,Fecha_Pedido,Hora_Pedido,Hora_Listo,Estado_Retiro,Local
MP-TAL-1021,Rodrigo,Torres,+56 9 9581 2235,Presencial,Prosciutto e Funghi,2026-08-30,12:15:00,12:28:00,No retirado (perdido),Talca
MP-TAL-1050,Soledad,Alarcón,+56 9 7558 8814,PedidosYa,Napoletana,2026-08-30,19:09:00,19:29:00,No retirado (perdido),Talca
MP-TAL-1007,Constanza,Riquelme,+56 9 8986 5010,Presencial,Pepperoni Clásica,2026-08-29,21:30:00,21:45:00,No retirado (perdido),Talca
MP-TAL-1038,Francisca,Araya,+56 9 6181 8144,Uber Eats,Napoletana,2026-08-29,15:21:00,15:38:00,No retirado (perdido),Talca
MP-TAL-1018,Fernanda,Guzmán,+56 9 6974 5562,Presencial,Prosciutto e Funghi,2026-08-28,21:50:00,22:09:00,No retirado (perdido),Talca
MP-TAL-1024,Francisco,Riquelme,+56 9 6249 6138,Uber Eats,Napoletana,2026-08-28,14:41:00,14:53:00,No retirado (perdido),Talca
MP-TAL-1040,Marcela,Espinoza,+56 9 8270 3085,Uber Eats,Cuatro Quesos,2026-08-28,21:42:00,21:58:00,No retirado (perdido),Talca
MP-TAL-1004,Florencia,Contreras,+56 9 7470 6635,Uber Eats,Vegetariana,2026-08-27,13:20:00,13:40:00,No retirado (perdido),Talca
MP-TAL-1046,Claudia,Vergara,+56 9 8289 6198,Uber Eats,Prosciutto e Funghi,2026-08-27,14:42:00,15:05:00,No retirado (perdido),Talca
MP-TAL-1022,Soledad,Torres,+56 9 6681 7658,PedidosYa,Prosciutto e Funghi,2026-08-25,18:28:00,18:40:00,No retirado (perdido),Talca


Total pedidos perdidos: 13


## ****Transformación: pedidos retirados OK (a tiempo o tarde, con minutos válidos)


In [0]:
df_retirados_ok = (
    df_raw
    .filter(col("Minutos_Espera_Retiro").isNotNull())
    .select(
        "ID_Pedido", "Nombre", "Apellido", "Telefono",
        "Canal", "Pizza", "Fecha_Pedido", "Hora_Pedido",
        "Hora_Listo", "Hora_Retiro", "Minutos_Espera_Retiro",
        "Estado_Retiro", "Local"
    )
    .orderBy(col("Minutos_Espera_Retiro").desc())
)

display(df_retirados_ok)
print("Total pedidos retirados:", df_retirados_ok.count())

ID_Pedido,Nombre,Apellido,Telefono,Canal,Pizza,Fecha_Pedido,Hora_Pedido,Hora_Listo,Hora_Retiro,Minutos_Espera_Retiro,Estado_Retiro,Local
MP-TAL-1035,Paulina,Hernández,+56 9 6817 8541,Presencial,Hawaiana,2026-08-27,17:36:00,17:52:00,18:30,38,Retirado tarde,Talca
MP-TAL-1042,Amanda,Fuentes,+56 9 8093 8752,Presencial,Pepperoni Clásica,2026-08-28,17:43:00,17:56:00,18:33,37,Retirado tarde,Talca
MP-TAL-1053,Nicolás,Díaz,+56 9 7065 7211,Presencial,Hawaiana,2026-08-27,16:44:00,17:08:00,17:45,37,Retirado tarde,Talca
MP-TAL-1033,Sergio,Torres,+56 9 8731 2684,PedidosYa,Hawaiana,2026-08-25,14:46:00,15:10:00,15:46,36,Retirado tarde,Talca
MP-TAL-1059,Álvaro,Contreras,+56 9 7775 6934,Uber Eats,Prosciutto e Funghi,2026-08-29,16:36:00,16:57:00,17:33,36,Retirado tarde,Talca
MP-TAL-1057,Francisco,Muñoz,+56 9 9804 6111,Presencial,BBQ Pollo,2026-08-29,16:08:00,16:21:00,16:54,33,Retirado tarde,Talca
MP-TAL-1006,Javiera,Tapia,+56 9 7138 8428,Uber Eats,Hawaiana,2026-08-29,13:13:00,13:34:00,14:05,31,Retirado tarde,Talca
MP-TAL-1012,Cristóbal,Espinoza,+56 9 7780 3591,PedidosYa,Prosciutto e Funghi,2026-08-28,13:48:00,14:13:00,14:44,31,Retirado tarde,Talca
MP-TAL-1029,Camila,Pizarro,+56 9 7226 2697,PedidosYa,Pepperoni Clásica,2026-08-25,16:38:00,16:54:00,17:24,30,Retirado tarde,Talca
MP-TAL-1045,Catalina,Rodríguez,+56 9 9293 9486,Presencial,Prosciutto e Funghi,2026-08-27,14:16:00,14:40:00,15:05,25,Retirado tarde,Talca


Total pedidos retirados: 47


## **Celda 6 — Load: guardar ambas como tablas Delta separadas**

In [0]:
df_perdidos.write.mode("overwrite").saveAsTable("melt_pizza.mel_pizza_raw.pedidos_perdidos")
df_retirados_ok.write.mode("overwrite").saveAsTable("melt_pizza.mel_pizza_raw.pedidos_retirados_ok")